In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    

In [ ]:
x = torch.rand(3, 3).to("cuda")
print(x.device)

In [ ]:
import boto3

session = boto3.Session()
print(session.get_credentials())

In [ ]:
import boto3

sts = boto3.client("sts")
print(sts.get_caller_identity())

In [ ]:
import os
import tarfile
import boto3
import torch
import torch.nn as nn
from torch import optim
from torchvision.models import vit_b_32
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

def create_vit_model(num_encoder_layers=2, num_classes=2):
    model = vit_b_32(weights=None)
    model.encoder.layers = nn.Sequential(
        *[model.encoder.layers[i] for i in range(num_encoder_layers)]
    )
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
bucket = "vit-stock-data-ikenna14"

s3_tar_key = "vit-data/vit_full_dataset.tar"

BASE_DIR = "/mnt/sagemaker-nvme/vit_full_dataset"

tar_path = "/mnt/sagemaker-nvme/vit_full_dataset.tar"

os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
# Block 3 — download tar from S3

s3 = boto3.client("s3") 

if not os.path.exists(tar_path):
    s3.download_file(
        bucket,
        s3_tar_key,
        tar_path
    )

In [ ]:
with tarfile.open(tar_path, 'r') as t:
    t.extractall(BASE_DIR)

print("Extract complete")

In [ ]:
import os

up_path = f"{BASE_DIR}/Train/up"
down_path = f"{BASE_DIR}/Train/down"

print("Up images:", len(os.listdir(up_path)))
print("Down images:", len(os.listdir(down_path)))

In [ ]:
print(os.listdir(f"{BASE_DIR}/Train/up")[:5])
print(os.listdir(f"{BASE_DIR}/Train/down")[:5])

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(
    root=f"{BASE_DIR}/Train",
    transform=transform
)





In [ ]:
from torch.utils.data import random_split

full_size = len(full_dataset)

train_size = int(0.7 * full_size)
val_size = full_size - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

In [ ]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Classes:", full_dataset.classes)
print("Total samples:", len(train_dataset))

In [ ]:

model = create_vit_model(num_encoder_layers=2, num_classes=2)
model = model.to(device)

print("Model ready on:", device)

In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("Loss and optimizer ready")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 1000 == 0:
            print(f"Train Batch {batch_idx} | Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
num_epochs = 10
best_val_acc = 0.0

save_path = "best_vit_model_full_scratch.pth"
checkpoint_path = "checkpoint.pth"

start_epoch = 0

In [ ]:
import os

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])

    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']

    print(f"Resuming from epoch {start_epoch}")

In [ ]:
for epoch in range(start_epoch, num_epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model to {save_path}")

    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_acc': best_val_acc
    }, checkpoint_path)

print(f"Best validation accuracy: {best_val_acc:.4f}")